# RSICCformer: Colab training and evaluation

This notebook clones the code from GitHub, prepares LEVIR-CC, and runs training, evaluation, and example caption generation. Code and data stay under `/content/RSICC`. Each run gets a separate Google Drive folder for checkpoints, settings, environment information, and results.

Select a GPU runtime in Colab. An earlier version completed training on an A100; the workflow changes in this notebook have not yet been tested in Colab. Adjust batch size and data-loading workers for your hardware.

Run the cells in order. To continue an earlier training run, set `RESUME_CHECKPOINT` in section 7. Only load checkpoints you trust: RSICC checkpoints contain serialized Python objects.

## 1. Check GPU and Java

Java is required for the METEOR metric.

In [ ]:
import torch
import subprocess

subprocess.run(["nvidia-smi"], check=True)
subprocess.run(["java", "-version"], check=True)
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime in Colab.")
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"GPU: {gpu_name} ({vram_gb:.1f} GB VRAM)")

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Clone the repository

Set `REPO_URL` and `REPO_REF` to the fork and branch, tag, or commit you want to run. The default branch below must be pushed to GitHub before Colab can use it. After merging these changes, you can use `main`; use a commit hash to reproduce a specific code version.

Start a new runtime if `/content/RSICC` already exists. After a runtime restart, restore the imports and configuration variables from these cells before continuing.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/yakupgulcan/RSICC.git"
REPO_REF = "experiment/colab-workflow"  # Branch, tag, or commit hash
PROJECT_DIR = Path("/content/RSICC")

if PROJECT_DIR.exists():
    raise FileExistsError(f"{PROJECT_DIR} already exists. Use a fresh runtime.")
subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)
subprocess.run(["git", "checkout", REPO_REF], cwd=PROJECT_DIR, check=True)
os.chdir(PROJECT_DIR)
subprocess.run(["git", "rev-parse", "HEAD"], check=True)

## 4. Install dependencies and create a run folder

Colab software versions can change between sessions. If installation requests a runtime restart, restart, restore the imports and project directory, then continue here.

Each execution of the next configuration cell creates a new folder named with a UTC timestamp and a random suffix. Run it once per experiment. Checkpoints and results from earlier experiments stay in their own folders. The preprocessed dataset cache is shared across runs.

In [ ]:
%pip install -r requirements.txt --quiet
%pip install gdown google-api-python-client google-auth --quiet

In [ ]:
import json
import os
import platform
import subprocess
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path
from uuid import uuid4
import torch
import torchvision

PROJECT_DIR = Path("/content/RSICC")
os.chdir(PROJECT_DIR)
OUTPUT_ROOT = Path("/content/drive/MyDrive/RSICC_Outputs")
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid4().hex[:8]
RUN_DIR = OUTPUT_ROOT / "runs" / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)
CHECKPOINT_DIR = PROJECT_DIR / "runs" / RUN_ID / "models_checkpoint"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=False)

environment = {
    "run_id": RUN_ID,
    "python": platform.python_version(),
    "pytorch": torch.__version__,
    "torchvision": torchvision.__version__,
    "gdown": version("gdown"),
    "pytorch_cuda_build": torch.version.cuda,
    "cuda_available": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "git_commit": subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip(),
    "repo_url": subprocess.check_output(["git", "remote", "get-url", "origin"], text=True).strip(),
}
(RUN_DIR / "environment.json").write_text(json.dumps(environment, indent=2))
print(json.dumps(environment, indent=2))
print(f"Run folder: {RUN_DIR}")

### Checkpoint storage

Authorize Drive API access. Large checkpoints stay on runtime storage and are uploaded with verified replacement, avoiding deleted checkpoint accumulation in Drive trash. Only previous checkpoint IDs in this run folder are managed; no general trash cleanup is performed. See [Drive files.delete](https://developers.google.com/workspace/drive/api/reference/rest/v3/files/delete).


In [ ]:
from google.colab import auth
from drive_checkpoints import DriveCheckpoints, drive_service

auth.authenticate_user()
service = drive_service()

def resolve_drive_folder(path, create=False):
    relative = Path(path).relative_to(Path("/content/drive/MyDrive"))
    folder_id = "root"
    for part in relative.parts:
        escaped = part.replace("\\", "\\\\").replace("'", "\\'")
        matches = service.files().list(
            q=f"'{folder_id}' in parents and name = '{escaped}' and mimeType = 'application/vnd.google-apps.folder' and trashed = false",
            fields="files(id)", pageSize=100,
        ).execute().get("files", [])
        if len(matches) > 1:
            raise RuntimeError(f"Multiple Drive folders named {part}; use a unique path.")
        if matches:
            folder_id = matches[0]["id"]
        elif create:
            folder_id = service.files().create(
                body={"name": part, "mimeType": "application/vnd.google-apps.folder", "parents": [folder_id]},
                fields="id",
            ).execute()["id"]
        else:
            raise FileNotFoundError(path)
    return folder_id

DRIVE_CHECKPOINT_DIR = RUN_DIR / "models_checkpoint"
DRIVE_CHECKPOINT_FOLDER_ID = resolve_drive_folder(DRIVE_CHECKPOINT_DIR, create=True)
print(f"Local checkpoints: {CHECKPOINT_DIR}")
print(f"Drive checkpoint folder ID: {DRIVE_CHECKPOINT_FOLDER_ID}")


## 5. Download LEVIR-CC

Set `DATASET_DRIVE_URL` to a dataset ZIP file. The extracted dataset must contain `LevirCCcaptions.json` and `images/`. Both are checked before skipping the download. A ZIP with a single enclosing dataset folder is also supported.

The command adds `--fuzzy` when the installed gdown supports it. Starting with gdown 6.1, share links are handled automatically and the flag has been removed. See the [gdown release notes](https://github.com/wkentaro/gdown/releases/tag/v6.1.0).

In [ ]:
import shutil
import sys
import zipfile

DATASET_DRIVE_URL = "https://drive.google.com/file/d/1YgWES-0OOL-3KK4yIV3-y8MN-_c1ELEE/view?usp=drive_link"
DATASET_DIR = PROJECT_DIR / "Levir_CC_dataset"
DATASET_DIR.mkdir(exist_ok=True)

def dataset_ready():
    return ((DATASET_DIR / "LevirCCcaptions.json").is_file()
            and (DATASET_DIR / "images" / "A").is_dir()
            and (DATASET_DIR / "images" / "B").is_dir())

if not dataset_ready():
    archive = Path("/content/dataset_download.zip")
    help_text = subprocess.check_output([sys.executable, "-m", "gdown", "--help"], text=True)
    command = [sys.executable, "-m", "gdown"]
    if "--fuzzy" in help_text:
        command.append("--fuzzy")
    subprocess.run(command + [DATASET_DRIVE_URL, "-O", str(archive)], check=True)
    with zipfile.ZipFile(archive) as z:
        z.extractall(DATASET_DIR)
    if not dataset_ready():
        candidates = [p.parent for p in DATASET_DIR.rglob("LevirCCcaptions.json")
                      if (p.parent / "images" / "A").is_dir()
                      and (p.parent / "images" / "B").is_dir()]
        if len(candidates) == 1 and candidates[0] != DATASET_DIR:
            source = candidates[0]
            shutil.copy2(source / "LevirCCcaptions.json", DATASET_DIR)
            shutil.copytree(source / "images", DATASET_DIR / "images", dirs_exist_ok=True)
    if not dataset_ready():
        raise FileNotFoundError("Expected LevirCCcaptions.json and images/A and images/B in the dataset ZIP.")
    archive.unlink()
else:
    print("Dataset already exists; skipping download.")
print(f"Dataset: {DATASET_DIR}")

## 6. Preprocess the data

Restore the shared `DRIVE_DATA_BACKUP` if available; otherwise run `create_input_files.py` and save a backup. The backup must use the same dataset, vocabulary, and preprocessing settings. Use a different cache path if those settings change. This keeps the existing backup location compatible with earlier runs.

In [ ]:
DATA_DIR = PROJECT_DIR / "data"
DATA_DIR.mkdir(exist_ok=True)
DATA_NAME = "LEVIR_CC_5_cap_per_img_5_min_word_freq"
DRIVE_DATA_BACKUP = Path("/content/drive/MyDrive/RSICC/data_preprocessed.zip")
required_files = [DATA_DIR / f"WORDMAP_{DATA_NAME}.json"]
for split in ("TRAIN", "VAL", "TEST"):
    required_files.extend(DATA_DIR / f"{split}_{kind}_{DATA_NAME}.{extension}"
                          for kind, extension in (("IMAGES", "hdf5"), ("CAPTIONS", "json"), ("CAPLENS", "json")))

if not all(p.is_file() for p in required_files):
    if DRIVE_DATA_BACKUP.is_file():
        print("Restoring preprocessed data...")
        with zipfile.ZipFile(DRIVE_DATA_BACKUP) as z:
            z.extractall(PROJECT_DIR)
    else:
        if any(DATA_DIR.iterdir()):
            raise RuntimeError("Incomplete data directory. Remove partial preprocessing outputs before retrying.")
        subprocess.run([sys.executable, "create_input_files.py"], check=True)
        if not all(p.is_file() for p in required_files):
            raise RuntimeError("Preprocessing did not create all required files.")
        DRIVE_DATA_BACKUP.parent.mkdir(parents=True, exist_ok=True)
        archive = shutil.make_archive("/content/data_preprocessed", "zip", PROJECT_DIR, "data")
        shutil.copy2(archive, DRIVE_DATA_BACKUP)
        Path(archive).unlink()
if not all(p.is_file() for p in required_files):
    raise RuntimeError("The preprocessed backup is incomplete or uses a different data name.")
print(f"Preprocessed data: {DATA_DIR}")

## 7. Train or resume training

The defaults use 20 epochs, batch size 32, and 4 data-loading workers. Reduce the batch size if GPU memory is insufficient. These values do not change the model architecture.

For a new training run, leave `RESUME_CHECKPOINT = None`. To continue, provide the full Drive path to an earlier checkpoint. The regular `checkpoint_...pth.tar` stores the latest completed epoch; `BEST_checkpoint_...pth.tar` stores the best validation BLEU-4 checkpoint. Older runs may only have the best checkpoint, so resuming those runs starts after that checkpoint's epoch.

`TOTAL_EPOCHS` is the total target, not the number of additional epochs. For example, after 20 completed epochs, set it to 30 for up to 10 more. Model weights, optimizer states, best BLEU-4, and the early-stopping counter are restored. New checkpoints also restore scheduler state. Legacy checkpoints lack scheduler state; counters restart (the existing StepLR uses gamma=1). Random-number and data-loader states are not restored, so this is not a bit-for-bit continuation.

Keep the same dataset, vocabulary, and model settings. Restored optimizer learning rates take precedence over the values below. Early stopping remains active: to continue a run that already exhausted patience, set `STOP_CRITERIA` above its saved counter. A resumed run writes into the new run folder and leaves its source checkpoint intact. Checkpoints are saved locally, then uploaded through the Drive API. Only the latest and best checkpoint files are retained in each run folder. The previous file of the same name is permanently deleted only after verifying the new upload size and MD5 checksum. Upload or deletion errors stop training. Keep space for old and new copies during upload. Existing Drive trash is not removed. Do not run concurrent training processes against the same checkpoint folder. If no new best score is reached, select the old best checkpoint explicitly for evaluation.

In [ ]:
RESUME_CHECKPOINT = None  # Example: "/content/drive/MyDrive/RSICC_Outputs/runs/<run_id>/models_checkpoint/checkpoint_resnet101_MCCFormers_diff_as_Q_trans.pth.tar"
TOTAL_EPOCHS = 20
BATCH_SIZE = 32
NUM_WORKERS = 4
STOP_CRITERIA = 10
ENCODER_LR = 1e-4
DECODER_LR = 1e-4

command = [
    sys.executable, "train.py",
    "--data_folder", str(DATA_DIR), "--data_name", DATA_NAME,
    "--savepath", str(CHECKPOINT_DIR),
    "--drive_checkpoint_folder_id", DRIVE_CHECKPOINT_FOLDER_ID,
    "--epochs", str(TOTAL_EPOCHS), "--batch_size", str(BATCH_SIZE),
    "--workers", str(NUM_WORKERS), "--stop_criteria", str(STOP_CRITERIA),
    "--encoder_lr", str(ENCODER_LR), "--decoder_lr", str(DECODER_LR),
]
if RESUME_CHECKPOINT is not None:
    if not Path(RESUME_CHECKPOINT).is_file():
        raise FileNotFoundError(RESUME_CHECKPOINT)
    source = Path(RESUME_CHECKPOINT)
    if source.is_relative_to(Path("/content/drive/MyDrive")):
        source_folder_id = resolve_drive_folder(source.parent, create=False)
        restored = CHECKPOINT_DIR / "resume_source" / source.name
        DriveCheckpoints(source_folder_id, service).download(source.name, restored)
        source = restored
    command += ["--checkpoint", str(source)]

config = {
    "dataset_url": DATASET_DRIVE_URL, "data_name": DATA_NAME,
    "preprocessed_backup": str(DRIVE_DATA_BACKUP),
    "resume_checkpoint": RESUME_CHECKPOINT,
    "drive_checkpoint_folder_id": DRIVE_CHECKPOINT_FOLDER_ID,
    "total_epochs": TOTAL_EPOCHS, "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS, "stop_criteria": STOP_CRITERIA,
    "encoder_lr": ENCODER_LR, "decoder_lr": DECODER_LR,
    "command": command,
}
(RUN_DIR / "training_config.json").write_text(json.dumps(config, indent=2))
subprocess.run(command, check=True)

## 8. Evaluate one checkpoint on the test set

Choose a single file with `EVAL_CHECKPOINT`. By default, this selects the best checkpoint from the current run. You can select the latest checkpoint or a checkpoint from an earlier run instead.

The scripts read every file in their checkpoint directory. This cell creates a temporary directory containing only the selected checkpoint. Each evaluation gets its own results folder, including the selected checkpoint path and command.

In [ ]:
import tempfile

CHECKPOINT_NAME = "checkpoint_resnet101_MCCFormers_diff_as_Q_trans.pth.tar"
EVAL_CHECKPOINT = CHECKPOINT_DIR / ("BEST_" + CHECKPOINT_NAME)
if not Path(EVAL_CHECKPOINT).is_file():
    raise FileNotFoundError(f"Select an existing checkpoint: {EVAL_CHECKPOINT}")

EVAL_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid4().hex[:8]
DRIVE_RESULTS_DIR = RUN_DIR / "eval_results" / EVAL_ID
DRIVE_RESULTS_DIR.mkdir(parents=True, exist_ok=False)
with tempfile.TemporaryDirectory(prefix="rsicc_eval_", dir="/content") as staging:
    os.symlink(Path(EVAL_CHECKPOINT).resolve(), Path(staging) / Path(EVAL_CHECKPOINT).name)
    command = [sys.executable, "eval.py", "--data_folder", str(DATA_DIR),
               "--data_name", DATA_NAME, "--path", staging, "--Split", "TEST", "--beam_size", "1"]
    (DRIVE_RESULTS_DIR / "evaluation_config.json").write_text(json.dumps(
        {"checkpoint": str(EVAL_CHECKPOINT), "command": command}, indent=2))
    subprocess.run(command, check=True)
shutil.copytree(PROJECT_DIR / "eval_results_fortest" / "TEST", DRIVE_RESULTS_DIR / "TEST")
print(f"Evaluation results: {DRIVE_RESULTS_DIR}")

## 9. Generate a caption for an example image pair

This uses the single checkpoint selected above. The generated caption is also copied to this run's Drive folder.

In [ ]:
if not Path(EVAL_CHECKPOINT).is_file():
    raise FileNotFoundError(EVAL_CHECKPOINT)
with tempfile.TemporaryDirectory(prefix="rsicc_caption_", dir="/content") as staging:
    os.symlink(Path(EVAL_CHECKPOINT).resolve(), Path(staging) / Path(EVAL_CHECKPOINT).name)
    subprocess.run([sys.executable, "caption.py",
                    "--img_A", "./Example/A/train_000016.png",
                    "--img_B", "./Example/B/train_000016.png",
                    "--path", staging, "--data_folder", str(DATA_DIR),
                    "--data_name", DATA_NAME], check=True)
CAPTION_DIR = RUN_DIR / "example_captions" / (datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid4().hex[:8])
CAPTION_DIR.mkdir(parents=True, exist_ok=False)
(CAPTION_DIR / "caption_config.json").write_text(json.dumps(
    {"checkpoint": str(EVAL_CHECKPOINT), "img_A": "Example/A/train_000016.png",
     "img_B": "Example/B/train_000016.png"}, indent=2))
for path in [PROJECT_DIR / "eval_results" / "resnet101_MCCFormers_diff_as_Q_trans_res.json"]:
    shutil.copy2(path, CAPTION_DIR)
    print(path.read_text())
print(f"Example captions: {CAPTION_DIR}")